# Spark Declarative Pipelines
## Declare your tables in SQL. Spark builds, orders, and validates the pipeline.

An SDP pipeline is just a **spec file** + some **transformation files**. We'll build a
bronze → silver → gold medallion over real food-delivery orders and land it in **Unity Catalog**.

_Re-running this? Run `!python3 /home/jovyan/demos/sdp-medallion/reset.py` first — UC has no TRUNCATE._

---
## 1. The whole pipeline is a handful of files
One Python file ingests the raw data (SDP can't read external files from SQL); everything else is plain SQL.

In [ ]:
!ls -1 /home/jovyan/demos/sdp-medallion/transformations

Here's the **silver** layer — parse the JSON body, derive time features, join the city dimension. Pure declarative SQL; SDP infers the dependency on `orders_bronze` and `dim_locations` from the `FROM`/`JOIN`:

In [ ]:
from IPython.display import Code
Code(filename='/home/jovyan/demos/sdp-medallion/transformations/silver_orders_enriched.sql')

---
## 2. SDP validates the whole DAG *before* running
With imperative code, a typo in a table name blows up at runtime — maybe deep into a job. SDP checks the dependency graph with a **dry-run** first. Here's a pipeline with a deliberate bug:

In [ ]:
Code(filename='/home/jovyan/demos/sdp-medallion/broken/transformations/gold_revenue.sql')

Run a **dry-run** — no data moves, it just validates the graph:

In [ ]:
!spark-pipelines dry-run --spec /home/jovyan/demos/sdp-medallion/broken/spark-pipeline.yml 2>&1 | grep -iE 'NOT_FOUND|Failed to resolve|orderz' || true

**Caught it** — `orderz` doesn't exist, with the exact line, before touching any data. Fix the typo and the graph validates clean. That's the point of declaring your dependencies.

---
## 3. Run the real medallion → Unity Catalog
SDP runs the layers in dependency order and registers catalog-managed Delta tables in your schema:

In [ ]:
!spark-pipelines run --spec /home/jovyan/demos/sdp-medallion/spark-pipeline.yml 2>&1 | grep -E 'is COMPLETED|Run is COMPLETED'

---
## 4. The result
Query the gold layer straight from Unity Catalog and chart it:

In [ ]:
import os
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
NS = os.environ.get('DEMO_NS','medallion_demo').rstrip('_') or 'medallion_demo'
bs = spark.table(f'managed_demo.{NS}.gold_brand_summary').orderBy('total_revenue', ascending=False).toPandas()
bs

In [ ]:
import matplotlib.pyplot as plt
top = bs.head(10)[::-1]
plt.figure(figsize=(8,4)); plt.barh(top['brand_name'], top['total_revenue'])
plt.xlabel('revenue'); plt.title(f'Revenue by brand  (managed_demo.{NS})'); plt.tight_layout(); plt.show()

---
### Recap
- The transforms were plain SQL `SELECT`s — SDP inferred the DAG, ordered execution, and handled the writes.
- `dry-run` caught a bad dependency before any data moved.
- The gold tables are now in Unity Catalog under your schema — open the UC console to browse them.